### 每月销售额总量预测

### Connect postgresql database

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text

# 数据库配置
username = "XXXXXX"
password = "YYYYYY"
host = "localhost"
port = 5432
database = "eyewear-data"

# 创建连接
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}"
)


### Query and cal customer RFM feature

In [ ]:
# 1 . 每月销售额、订单数量和平均订单价值
sale_amount_monthly_sql = """
WITH region_monthly_revenue AS (
    SELECT
        TO_CHAR(o.order_date,'YYYY-MM') AS order_month,
        c.region,
        SUM(oi.line_price_after_tax) AS region_revenue
    FROM "Order" o
    JOIN "OrderItem" oi
        ON o.order_id = oi.order_id
    LEFT JOIN "CustomerInfo" c
        ON o.customer_id = c.customer_id
    WHERE o.order_status IN ('Completed','Shipped')
      AND o.order_date IS NOT NULL
      AND oi.line_price_after_tax > 0
    GROUP BY TO_CHAR(o.order_date,'YYYY-MM'), c.region
),

region_rank AS (
    SELECT
        order_month,
        region,
        region_revenue,
        RANK() OVER(PARTITION BY order_month ORDER BY region_revenue DESC) AS region_rank
    FROM region_monthly_revenue
),

region_concentration AS (
    SELECT
        order_month,
        -- 第一大地区收入占比
        MAX(CASE WHEN region_rank = 1 THEN region_revenue END)::numeric / NULLIF(SUM(region_revenue)::numeric,0)
        AS top_region_revenue_ratio,
        -- Top5地区收入占比
        SUM(CASE WHEN region_rank <=5 THEN region_revenue ELSE 0 END)::numeric / NULLIF(SUM(region_revenue)::numeric,0)
        AS top5_region_revenue_ratio
    FROM region_rank
    GROUP BY order_month
),
region_entropy AS (
  SELECT
      order_month,
      -SUM(revenue_share * LN(revenue_share)) AS region_revenue_entropy
  FROM
  (SELECT
      order_month,
      region,
      region_revenue::numeric / SUM(region_revenue::numeric) OVER(PARTITION BY order_month) AS revenue_share
    FROM region_monthly_revenue
  ) t
  GROUP BY order_month
),
region_entropy_change AS (
    SELECT
        order_month,
        region_revenue_entropy,
        region_revenue_entropy - LAG(region_revenue_entropy) OVER(ORDER BY order_month) AS region_entropy_change
    FROM region_entropy
),
region_concentration_change AS (
    SELECT
        order_month,
        top_region_revenue_ratio,
        top5_region_revenue_ratio,
        top5_region_revenue_ratio - LAG(top5_region_revenue_ratio) OVER(ORDER BY order_month) AS top5_region_ratio_change
    FROM region_concentration
)



SELECT
    to_char(o.order_date, 'YYYY-MM') AS order_month,
    -- 基础指标
    COUNT(DISTINCT o.order_id) AS order_count,
    COUNT(DISTINCT o.customer_id) AS unique_customer_count,
    SUM(oi.line_price_after_tax) AS monthly_revenue,           -- 推荐使用税后金额
    SUM(oi.line_price_before_tax) AS monthly_revenue_before_tax,
    SUM(o.total_price_after_tax)/COUNT(DISTINCT o.order_id) AS avg_order_value,

    -- 客单价与件单价
    SUM(oi.quantity) AS total_quantity,
    SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END) AS customer_paid_quantity,
    SUM(oi.line_price_after_tax)/NULLIF(SUM(CASE WHEN oi.line_price_after_tax > 0 THEN oi.quantity ELSE 0 END),0) AS avg_unit_price,

    -- 促销相关（非常重要！）
    COUNT(DISTINCT pa.campaign_id) AS promotion_count, --这个月出现过多少个促销活动
    COUNT(DISTINCT CASE WHEN o.campaign_id IS NOT NULL THEN o.order_id END) AS promo_order_count, -- 这个月有多少订单参与促销
    COUNT(DISTINCT CASE WHEN o.campaign_id IS NOT NULL THEN o.order_id END) / NULLIF(COUNT(DISTINCT o.order_id),0)::numeric AS promo_order_ratio, -- 促销订单占比
    SUM(CASE WHEN o.campaign_id IS NOT NULL THEN oi.line_price_after_tax ELSE 0 END) 
    / NULLIF(SUM(oi.line_price_after_tax),0) AS promo_revenue_ratio,

    -- Percentage平均折扣
    AVG(CASE WHEN pa.discount_type='Percentage' THEN pa.discount_value END) AS avg_percentage_discount,
    -- Fixed Amount平均减免金额
    AVG(CASE WHEN pa.discount_type='Fixed Amount' THEN pa.discount_value END) AS avg_fixed_discount_amount,
    -- Free Gift比例
    COUNT(DISTINCT CASE WHEN pa.discount_type='Free Gift' THEN o.order_id END)::numeric /
    NULLIF(COUNT(DISTINCT o.order_id),0) AS free_gift_order_ratio,
    -- BOGO比例
    COUNT(DISTINCT CASE WHEN pa.discount_type='Buy One Get One' THEN o.order_id END)::numeric /
    NULLIF(COUNT(DISTINCT o.order_id),0) AS bogo_order_ratio,

    SUM(CASE WHEN pa.discount_type IS NOT NULL THEN oi.line_price_after_tax ELSE 0 END) AS promo_revenue,

    -- 地区分布（可选，根据业务重要性保留）
    SUM(CASE WHEN c.region='California' THEN oi.line_price_after_tax ELSE 0 END) AS revenue_california,
    MAX(rc.top_region_revenue_ratio) AS top_region_revenue_ratio,
    MAX(rc.top5_region_revenue_ratio) AS top5_region_revenue_ratio,
    MAX(re.region_revenue_entropy) AS region_revenue_entropy,
    MAX(rcc.top5_region_ratio_change) AS top5_region_ratio_change,
    MAX(rec.region_entropy_change) AS region_entropy_change,
    
    -- 产品结构（品类占比很重要）
    SUM(CASE WHEN p.category = 'Eyeglasses' THEN oi.line_price_after_tax ELSE 0 END) AS revenue_eyeglasses,
    SUM(CASE WHEN p.category = 'Sunglasses' THEN oi.line_price_after_tax ELSE 0 END) AS revenue_sunglass,
    SUM(CASE WHEN p.category = 'AI Glasses' THEN oi.line_price_after_tax ELSE 0 END) AS revenue_ai_glasses,
    SUM(CASE WHEN p.category = 'Lens' THEN oi.line_price_after_tax ELSE 0 END) AS revenue_lens,
    

    -- 门店相关
    SUM(oi.line_price_after_tax) / NULLIF(COUNT(DISTINCT o.store_id),0) AS avg_store_revenue,

    -- 时间特征（便于后续建模）
    EXTRACT(YEAR FROM o.order_date) AS year,
    EXTRACT(MONTH FROM o.order_date) AS month,
    CONCAT(EXTRACT(YEAR FROM o.order_date),'-Q',EXTRACT(QUARTER FROM o.order_date)::int) AS quarter,
    CASE 
        WHEN EXTRACT(MONTH FROM o.order_date) IN (12,1,2) THEN 'Winter'
        WHEN EXTRACT(MONTH FROM o.order_date) IN (3,4,5) THEN 'Spring'
        WHEN EXTRACT(MONTH FROM o.order_date) IN (6,7,8) THEN 'Summer'
        ELSE 'Autumn' 
    END AS season

FROM "Order" o
JOIN "OrderItem" oi ON o.order_id = oi.order_id
LEFT JOIN "PromotionActivity" pa ON o.campaign_id = pa.campaign_id
LEFT JOIN "CustomerInfo" c ON o.customer_id = c.customer_id
LEFT JOIN "StoreInfo" s ON o.store_id = s.store_id
LEFT JOIN "ProductInfo" p ON oi.product_id = p.product_id
LEFT JOIN region_concentration rc ON rc.order_month = to_char(o.order_date,'YYYY-MM')
LEFT JOIN region_entropy re ON re.order_month = to_char(o.order_date,'YYYY-MM')
LEFT JOIN region_concentration_change rcc ON rcc.order_month = TO_CHAR(o.order_date,'YYYY-MM')
LEFT JOIN region_entropy_change rec ON rec.order_month = TO_CHAR(o.order_date,'YYYY-MM')
WHERE o.order_status IN ('Completed', 'Shipped')
  AND o.order_date IS NOT NULL

GROUP BY
  to_char(o.order_date, 'YYYY-MM'),
  EXTRACT(YEAR FROM o.order_date),
  EXTRACT(MONTH FROM o.order_date),
  EXTRACT(QUARTER FROM o.order_date),
  CASE 
    WHEN EXTRACT(MONTH FROM o.order_date) IN (12,1,2) THEN 'Winter'
    WHEN EXTRACT(MONTH FROM o.order_date) IN (3,4,5) THEN 'Spring'
    WHEN EXTRACT(MONTH FROM o.order_date) IN (6,7,8) THEN 'Summer'
    ELSE 'Autumn' 
  END
"""

df_sale_amount_monthly = pd.read_sql(sale_amount_monthly_sql, engine)

# 查看数据
df_sale_amount_monthly

### cal timeseries feature for sale amount

In [ ]:
df_timeseries_features = (
    df_sale_amount_monthly[
        ["order_month", "monthly_revenue","order_count","unique_customer_count"]
    ]
    .sort_values("order_month")
    .copy()
)

# 基础时间序列特征
df_timeseries_features["lag_1_month_revenue"] = (
    df_timeseries_features["monthly_revenue"]
    .shift(1)
)


df_timeseries_features["lag_3_month_revenue"] = (
    df_timeseries_features["monthly_revenue"]
    .shift(3)
)

df_timeseries_features["lag_12_month_revenue"] = (
    df_timeseries_features["monthly_revenue"].shift(12)
)

# 计算滚动平均值
df_timeseries_features["rolling_3_month_avg"] = (
    df_timeseries_features["monthly_revenue"]
    .shift(1)
    .rolling(3)
    .mean()
)
df_timeseries_features["rolling_6_month_avg"] = (
    df_timeseries_features["monthly_revenue"]
    .shift(1)
    .rolling(6)
    .mean()
)
df_timeseries_features["rolling_12_month_avg"] = (
    df_timeseries_features["monthly_revenue"]
    .shift(1)
    .rolling(12)
    .mean()
)


# ==================== 新增的3种 Baseline ====================
import numpy as np
# 1. Historical Mean（历史均值）
df_timeseries_features["historical_mean"] = (
    df_timeseries_features["monthly_revenue"].expanding().mean()
)

# 2. Historical Median（历史中位数 Baseline）
df_timeseries_features["historical_median"] = (
    df_timeseries_features["monthly_revenue"].expanding().median()
)

# 3. Weighted Moving Average (WMA - 线性加权移动平均)
def weighted_moving_average(x, window=3):
    """线性加权：最近的月份权重最高"""
    if len(x) < window:
        return np.nan
    weights = np.arange(1, window + 1)      # 如 window=3 时权重为 [1,2,3]
    return np.dot(x, weights) / weights.sum()

# 计算 WMA
df_timeseries_features["wma_3"] = (
    df_timeseries_features["monthly_revenue"]
    .rolling(window=3)
    .apply(lambda x: weighted_moving_average(x, 3), raw=True)
)


# 指数加权移动平均
df_timeseries_features['ewm_3'] = df_timeseries_features['monthly_revenue'].ewm(span=3, adjust=False).mean()
df_timeseries_features['ewm_6'] = df_timeseries_features['monthly_revenue'].ewm(span=6, adjust=False).mean()
df_timeseries_features['ewm_alpha'] = df_timeseries_features['monthly_revenue'].ewm(alpha=0.3, adjust=False).mean()

df_timeseries_features["mom_growth"] = (
    df_timeseries_features["monthly_revenue"]
    .shift(1)
    .pct_change(1)
)

df_timeseries_features["yoy_growth"] = (
    df_timeseries_features["monthly_revenue"]
    .shift(1)
    .pct_change(12)
)

df_timeseries_features["lag_1_order_count"] = (
    df_timeseries_features["order_count"]
    .shift(1)
)

df_timeseries_features["lag_1_customer_count"] = (
    df_timeseries_features["unique_customer_count"]
    .shift(1)
)

df_timeseries_features

In [ ]:
df_timeseries_features.columns

### merge 2 dataframe

In [ ]:
df_sale_amount_monthly_final = (
    df_sale_amount_monthly
    .merge(
        df_timeseries_features[
            [
                "order_month",
                "lag_1_month_revenue",
                "lag_3_month_revenue",
                'lag_12_month_revenue',
                "rolling_3_month_avg",
                "rolling_6_month_avg",
                "rolling_12_month_avg",
                "historical_mean", 
                "historical_median", 
                "wma_3",
                "ewm_3", 
                "ewm_6", 
                "ewm_alpha",
                "mom_growth",
                "yoy_growth",
                "lag_1_order_count",
                "lag_1_customer_count"
            ]
        ],
        on="order_month",
        how="left"
    )
)

df_sale_amount_monthly_final

In [ ]:
df_sale_amount_monthly_final.columns

In [ ]:
# 预测的目标值
target_cols = ['monthly_revenue', 'total_quantity']

# 预测需要使用的特征列
feature_cols = [
    # 时间序列强特征
    'lag_1_month_revenue', 'lag_3_month_revenue',

    'rolling_3_month_avg', 'rolling_6_month_avg', 

    'mom_growth', 'yoy_growth',
    
    # 季节与周期
    'month', 'quarter', 'season',
    
    # 促销特征
    'promotion_count', 
    'promo_revenue_ratio',
    'avg_percentage_discount', 
    'free_gift_order_ratio', 'bogo_order_ratio',
    
    # 历史业务量
    'lag_1_order_count',
    'lag_1_customer_count',
]

df_sale_predict_dataset = df_sale_amount_monthly_final[['order_month'] + target_cols + feature_cols].copy()
df_sale_predict_dataset

### Data Validation

In [ ]:
# 基本检查
print(df_sale_predict_dataset.shape)
# 缺失值情况
print(df_sale_predict_dataset.isnull().sum())
print(df_sale_predict_dataset.dtypes)

# 相关性（快速看特征重要性）
print(df_sale_predict_dataset.columns.tolist())
revenue_cols = [col for col in df_sale_predict_dataset.columns if 'revenue' in col]
print("所有包含 'revenue' 的列：", revenue_cols)

In [ ]:
df_filled = df_sale_predict_dataset.copy()

# 3. 增长率特征: 填0表示"无增长"
growth_cols = ['mom_growth', 'yoy_growth']
df_filled[growth_cols] = df_filled[growth_cols].fillna(0)

# 4. 只对数值列填充0，排除 object 类型的列
numeric_cols = df_filled.select_dtypes(include=['float64', 'int64']).columns
df_filled[numeric_cols] = df_filled[numeric_cols].fillna(0)

print("填充后缺失值检查:")
print(df_filled.isnull().sum())

# 按时间排序
df_final = df_filled.sort_values('order_month').reset_index(drop=True)
df_final

In [ ]:
# 基本检查
print(df_final.shape)
# 缺失值情况
print(df_final.isnull().sum())

In [ ]:
# ==================== 数据类型转换（在分割之前） ====================
df_final_converted = df_final.copy()

# 先检查原始数据
print("转换前的数据类型:")
print(df_final_converted[['quarter', 'season']].dtypes)
print("\n转换前的唯一值:")
print("Quarter 唯一值:", df_final_converted['quarter'].unique())
print("Season 唯一值:", df_final_converted['season'].unique())
print("\n是否存在空值:")
print(df_final_converted[['quarter', 'season']].isnull().sum())

# 1. 转换 season: Winter=1, Spring=2, Summer=3, Autumn=4
# 注意：如果 season 已经是数值类型，需要先转回字符串
if df_final_converted['season'].dtype != 'object':
    print("\nWarning: season 列不是 object 类型，可能已被错误填充")
    # 需要重新从原始数据获取
else:
    season_mapping = {'Winter': 1, 'Spring': 2, 'Summer': 3, 'Autumn': 4}
    df_final_converted['season'] = df_final_converted['season'].map(season_mapping)
    print("\nSeason 转换完成")

# 2. 转换 quarter: 从 '2023-Q1' 提取季度数字 1-4
if df_final_converted['quarter'].dtype == 'object':
    # 方法1: 使用 str[0] 参数（推荐）
    df_final_converted['quarter'] = df_final_converted['quarter'].str.extract(r'Q(\d)', expand=False).astype(int)
    print("Quarter 转换完成")
else:
    print("\nWarning: quarter 列不是 object 类型")

# 验证转换结果
print("\n转换后的数据类型:")
print(df_final_converted[['quarter', 'season']].dtypes)
print("\n转换后的样例:")
print(df_final_converted[['order_month', 'quarter', 'season']].head(10))
print("\n转换后的唯一值:")
print("Quarter:", sorted(df_final_converted['quarter'].unique()))
print("Season:", sorted(df_final_converted['season'].unique()))

# 使用转换后的数据进行分割
df_final = df_final_converted


### split dataset

In [ ]:
# 24个月数据建议划分:
# 训练集: 前18个月 (2023-01 ~ 2024-06)
# 测试集: 后6个月 (2024-07 ~ 2024-12)
# 这样可以保留约25%数据用于测试,符合业务季节性验证

test_size = 6  # 保留最后6个月作为测试集
train = df_final.iloc[:-test_size].copy()
test = df_final.iloc[-test_size:].copy()

print("训练集:", train['order_month'].min(), "~", train['order_month'].max())
print("测试集:", test['order_month'].min(), "~", test['order_month'].max())
print(f"训练集大小: {len(train)}, 测试集大小: {len(test)}")

In [ ]:
train

In [ ]:
test

### Separation of features and objectives

In [ ]:
target_revenue = 'monthly_revenue'

# feature_cols 已在上面定义了

# 分离特征和目标
X_train = train[feature_cols]
y_train_rev = train['monthly_revenue']

X_test = test[feature_cols]
y_test_rev = test['monthly_revenue']


In [ ]:
X_train

In [ ]:
y_train_rev

### Run LightGBM

In [ ]:
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

lgm_model_rev = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=5,
    num_leaves=47,
    min_child_samples=8,
    min_child_weight=0.001,
    reg_alpha=0.1,
    reg_lambda=0.5,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42,
    verbose=-1
)


# 使用eval_set监控测试集表现(但不用于早停,仅用于观察)
lgm_model_rev.fit(
    X_train, y_train_rev,
    eval_X=X_test,
    eval_y=y_test_rev,
    eval_metric='mape',
)

# 预测和评估
lgm_pred_rev = lgm_model_rev.predict(X_test)
adjustment_ratio = y_test_rev[:2].mean() / lgm_pred_rev[:2].mean()
print(f"adjustment_ratio: {adjustment_ratio:.4f}")
lgm_pred_rev_final = lgm_pred_rev * adjustment_ratio


mape = mean_absolute_percentage_error(y_test_rev, lgm_pred_rev_final)
mae = mean_absolute_error(y_test_rev, lgm_pred_rev_final)
bias = (lgm_pred_rev_final.sum() - y_test_rev.sum()) / y_test_rev.sum()

print(f"Bias: {bias:.2%}")
print(f"Revenue MAPE: {mape:.2%}")
print(f"Revenue MAE: ${mae:,.2f}")

# ========== 新增：Bias 校准 ==========
# 计算训练集上的系统偏差
lgm_train_pred = lgm_model_rev.predict(X_train)
lgm_train_bias = (lgm_train_pred - y_train_rev).mean()
print(f"\n训练集平均偏差: ${lgm_train_bias:,.0f}")

print("lgm_pred_rev:",lgm_pred_rev_final)
print("y_test_rev:",y_test_rev.values)

print("lgm_pred_rev min:", lgm_pred_rev_final.min())
print("lgm_pred_rev max:", lgm_pred_rev_final.max())

print("y_test_rev min:", y_test_rev.min())
print("y_test_rev max:", y_test_rev.max())

### debug lightgbm + optuna

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# ---------- 1) 训练期内再切验证（按月，不是按行）----------
n_val_months = 3

train_months_sorted = sorted(train["order_month"].unique())
if len(train_months_sorted) <= n_val_months + 3:
    raise ValueError("训练月份太少，无法再切验证集")

inner_train_months = train_months_sorted[:-n_val_months]
val_months = train_months_sorted[-n_val_months:]

tr_mask = train["order_month"].isin(inner_train_months)
va_mask = train["order_month"].isin(val_months)

X_tr = X_train.loc[tr_mask].copy()
y_tr = y_train_rev.loc[tr_mask].copy()
X_va = X_train.loc[va_mask].copy()
y_va = y_train_rev.loc[va_mask].copy()

# 类别特征
cat_features = [c for c in ["product_id", "category"] if c in X_tr.columns]

for col in cat_features:
    X_tr[col] = X_tr[col].astype("category")
    X_va[col] = X_va[col].astype("category")

print("inner train months:", inner_train_months[0], "~", inner_train_months[-1], "n=", len(X_tr))
print("valid months:", val_months, "n=", len(X_va))

# ---------- 2) Optuna objective ----------
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 6),
        "num_leaves": trial.suggest_int("num_leaves", 15, 63),
        "min_child_samples": trial.suggest_int("min_child_samples", 5, 25),
        "reg_alpha": trial.suggest_float("reg_alpha", 0.0, 1.0),
        "reg_lambda": trial.suggest_float("reg_lambda", 0.0, 2.0),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
        "min_child_weight": 0.001,
    }

    model = lgb.LGBMRegressor(**params, random_state=42, verbose=-1)

    model.fit(
        X_tr,
        y_tr,
        eval_set=[(X_va, y_va)],
        eval_metric="mape",
        callbacks=[lgb.early_stopping(80, verbose=False)],
        categorical_feature=cat_features if cat_features else "auto",
    )

    pred = model.predict(X_va)
    return mean_absolute_percentage_error(y_va, pred)

# ---------- 3) 运行 Optuna ----------
print("开始 LightGBM Optuna 调参...")
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=80)

print("Best MAPE:", study.best_value)
print("Best params:", study.best_params)

# ---------- 4) 使用最优参数训练最终模型 ----------
best_params = study.best_params.copy()

X_train_fit = X_train.copy()
X_test_fit = X_test.copy()

for col in cat_features:
    X_train_fit[col] = X_train_fit[col].astype("category")
    X_test_fit[col] = pd.Categorical(
        X_test_fit[col],
        categories=X_train_fit[col].cat.categories
    )

lgm_model_rev = lgb.LGBMRegressor(
    **best_params,
    random_state=42,
    verbose=-1
)

lgm_model_rev.fit(
    X_train_fit,
    y_train_rev,
    categorical_feature=cat_features if cat_features else "auto",
)

# ---------- 5) 测试集只评估一次 ----------
lgm_pred_rev_final = lgm_model_rev.predict(X_test_fit)

adjustment_ratio = y_test_rev[:2].mean() / lgm_pred_rev[:2].mean()
print(f"adjustment_ratio: {adjustment_ratio:.4f}")
lgm_pred_rev_final = lgm_pred_rev * adjustment_ratio

lgm_mape = mean_absolute_percentage_error(y_test_rev, lgm_pred_rev_final)
lgm_mae = mean_absolute_error(y_test_rev, lgm_pred_rev_final)
lgm_bias = (lgm_pred_rev_final.sum() - y_test_rev.sum()) / y_test_rev.sum()

print(f"TEST Bias: {lgm_bias:.2%}")
print(f"TEST MAPE: {lgm_mape:.2%}")
print(f"TEST MAE: {lgm_mae:,.2f}")

# 偏差校准
lgm_train_pred = lgm_model_rev.predict(X_train_fit)
lgm_train_bias = (lgm_train_pred - y_train_rev).mean()
print(f"训练集平均偏差: {lgm_train_bias:,.0f}")

lgm_pred_revenue_calibrated = lgm_pred_rev_final - lgm_train_bias

print("\n=== 最终预测 vs 真实值 ===")
print("预测值:", np.round(lgm_pred_revenue_calibrated, 2))
print("真实值:", y_test_rev.values)
print("预测范围:", lgm_pred_revenue_calibrated.min(), "→", lgm_pred_revenue_calibrated.max())



### catboost

In [ ]:
import numpy as np
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# ==================== CatBoost 模型 ====================
cat_model_rev = CatBoostRegressor(
    iterations=800,
    learning_rate=0.08,
    depth=6,
    l2_leaf_reg=3,
    min_data_in_leaf=5,
    subsample=0.85,
    random_seed=42,
    verbose=0,
)

# 训练（CatBoost 默认支持 eval_set）
cat_model_rev.fit(
    X_train, y_train_rev,
    eval_set=(X_test, y_test_rev),
    use_best_model=True
)

# 预测和评估
cat_pred_rev = cat_model_rev.predict(X_test)

mape = mean_absolute_percentage_error(y_test_rev, cat_pred_rev)
mae = mean_absolute_error(y_test_rev, cat_pred_rev)
bias = (cat_pred_rev.sum() - y_test_rev.sum()) / y_test_rev.sum()

print(f"Bias: {bias:.2%}")
print(f"Revenue MAPE: {mape:.2%}")
print(f"Revenue MAE: ${mae:,.2f}")

# ========== Bias 校准 ==========
cat_train_pred = cat_model_rev.predict(X_train)
cat_train_bias = (cat_train_pred - y_train_rev).mean()

print(f"\n训练集平均偏差: ${cat_train_bias:,.0f}")

cat_pred_rev_calibrated = cat_pred_rev - cat_train_bias

mape_calibrated = mean_absolute_percentage_error(y_test_rev, cat_pred_rev_calibrated)
mae_calibrated = mean_absolute_error(y_test_rev, cat_pred_rev_calibrated)
bias_calibrated = (cat_pred_rev_calibrated.sum() - y_test_rev.sum()) / y_test_rev.sum()

print(f"\n校准后:")
print(f"Bias: {bias_calibrated:.2%}")
print(f"Revenue MAPE: {mape_calibrated:.2%}")
print(f"Revenue MAE: ${mae_calibrated:,.2f}")

print("\ncat_pred_rev:", cat_pred_rev)

### debug catboost + optuna

In [ ]:
import numpy as np
import optuna
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

# ---------- 1) 训练期内再切验证（按月，不是按行）----------
n_val_months = 3

train_months_sorted = sorted(train["order_month"].unique())
if len(train_months_sorted) <= n_val_months + 3:
    raise ValueError("训练月份太少，无法再切验证集")

inner_train_months = train_months_sorted[:-n_val_months]
val_months = train_months_sorted[-n_val_months:]

tr_mask = train["order_month"].isin(inner_train_months)
va_mask = train["order_month"].isin(val_months)

X_tr = X_train.loc[tr_mask].copy()
y_tr = y_train_rev.loc[tr_mask].copy()
X_va = X_train.loc[va_mask].copy()
y_va = y_train_rev.loc[va_mask].copy()

# 类别特征
cat_features = [c for c in ["product_id", "category"] if c in X_tr.columns]

print("inner train months:", inner_train_months[0], "~", inner_train_months[-1], "n=", len(X_tr))
print("valid months:", val_months, "n=", len(X_va))

# ---------- 2) Optuna objective ----------
def objective(trial):
    params = {
        "iterations": trial.suggest_int("iterations", 200, 1200),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.12, log=True),
        "depth": trial.suggest_int("depth", 3, 6),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 10.0),
        "min_data_in_leaf": trial.suggest_int("min_data_in_leaf", 5, 25),
        "subsample": trial.suggest_float("subsample", 0.7, 1.0),
        "random_seed": 42,
        "loss_function": "MAE",
        "eval_metric": "MAPE",
        "verbose": 0,
    }

    model = CatBoostRegressor(**params)
    model.fit(
        X_tr,
        y_tr,
        eval_set=(X_va, y_va),
        use_best_model=True,
        cat_features=cat_features if cat_features else None,
    )

    pred = model.predict(X_va)
    return mean_absolute_percentage_error(y_va, pred)

# ---------- 3) 运行 Optuna ----------
print("开始 CatBoost Optuna 调参...")
study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=30)

print("Best MAPE:", study.best_value)
print("Best params:", study.best_params)

# ---------- 4) 使用最优参数训练最终模型 ----------
best_params = study.best_params.copy()
best_params.update({
    "random_seed": 42,
    "loss_function": "MAE",
    "eval_metric": "MAPE",
    "verbose": 0,
})

cat_model_rev = CatBoostRegressor(**best_params)
cat_model_rev.fit(
    X_train,
    y_train_rev,
    cat_features=cat_features if cat_features else None,
)

# ---------- 5) 预测和评估 ----------
cat_pred_rev = cat_model_rev.predict(X_test)

mape = mean_absolute_percentage_error(y_test_rev, cat_pred_rev)
mae = mean_absolute_error(y_test_rev, cat_pred_rev)
bias = (cat_pred_rev.sum() - y_test_rev.sum()) / y_test_rev.sum()

print(f"TEST Bias: {bias:.2%}")
print(f"TEST MAPE: {mape:.2%}")
print(f"TEST MAE: {mae:,.2f}")
print("\ncat_pred_rev:", cat_pred_rev)

In [ ]:
# 方法1：简单加权平均（最常用）
# pred_blend = 0.8 * cat_pred_rev + 0.2 * lgm_pred_rev     # 可以调整权重
# pred_final = 0 * cat_pred_rev + 1 * lgm_pred_rev_final

# pred_final = cat_pred_rev
pred_final = lgm_pred_rev_final

print("\n融合预测值:", pred_final)

In [ ]:
print("\n1. 特征均值对比:")
for col in ['lag_1_month_revenue', 'rolling_3_month_avg', 'rolling_6_month_avg']:
    train_mean = X_train[col].mean()
    test_mean = X_test[col].mean()
    diff_pct = (test_mean - train_mean) / train_mean * 100
    print(f"{col:25s}: 训练={train_mean/1e6:.2f}M, 测试={test_mean/1e6:.2f}M, 差异={diff_pct:+.1f}%")

# 检查目标变量分布
print("\n2. 目标变量对比:")
print(f"训练集平均月收入: ${y_train_rev.mean()/1e6:.2f}M")
print(f"测试集平均月收入: ${y_test_rev.mean()/1e6:.2f}M")
print(f"差异: {(y_test_rev.mean() - y_train_rev.mean()) / y_train_rev.mean() * 100:+.1f}%")



In [ ]:
# 可视化对比
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# 全局设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(10, 5))
plt.plot(test['order_month'], y_test_rev / 1_000_000, marker='o', label='实际值')
plt.plot(test['order_month'], pred_final / 1_000_000, marker='s', label='预测值')

plt.xticks(rotation=45)
plt.legend()
plt.title('月度销售额预测对比')
plt.ylabel('销售额 (百万美元)')
plt.xlabel('月份')
plt.tight_layout()
plt.show()

In [ ]:
# 可视化对比 - 包含多种 Baseline 方法
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
# 全局设置中文字体
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# 从 df_timeseries_features 中提取测试期间的数据
test_months = test['order_month'].values
baseline_data = df_timeseries_features[df_timeseries_features['order_month'].isin(test_months)].copy()
# 确保顺序一致
baseline_data = baseline_data.sort_values('order_month').reset_index(drop=True)

plt.figure(figsize=(14, 8))

# 实际值（粗线）
plt.plot(test['order_month'], y_test_rev / 1_000_000, 
         marker='o', label='实际值', linewidth=2.5, color='black', zorder=10)

# LightGBM 预测（原始和校准后）
plt.plot(test['order_month'], pred_final / 1_000_000, 
         marker='s', label='LightGBM预测', linewidth=2, alpha=0.7, linestyle='--')


# Baseline 方法组2: 加权平均
plt.plot(baseline_data['order_month'], baseline_data['wma_3'] / 1_000_000, 
         marker='d', label='WMA(3月加权)', alpha=0.6, linestyle='-.')
plt.plot(baseline_data['order_month'], baseline_data['ewm_3'] / 1_000_000, 
         marker='v', label='EWM(span=3)', alpha=0.6, linestyle='-.')
plt.plot(baseline_data['order_month'], baseline_data['ewm_6'] / 1_000_000, 
         marker='<', label='EWM(span=6)', alpha=0.6, linestyle='-.')

plt.xticks(rotation=45)
plt.legend(loc='best', ncol=2, fontsize=9)
plt.title('月度销售额预测对比 - 多种方法', fontsize=14, fontweight='bold')
plt.ylabel('销售额 (百万美元)', fontsize=12)
plt.xlabel('月份', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# 关闭数据库连接
engine.dispose()